In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
from utils import *
from fractions import Fraction

# Isotropic state

In [ ]:
def optimal_witness_isotropic(local_dim, schmidt_number):
    # Construct the optimal geometric Schmidt witness for the isotropic state
    tmp0 = numqi.state.maximally_entangled_state(local_dim)
    tmp1 = (schmidt_number - 1) / local_dim * np.eye(local_dim**2) - tmp0[:,np.newaxis] @ tmp0[np.newaxis,:]
    ret = (local_dim/np.sqrt(local_dim**2-1))*tmp1
    return ret

In [ ]:
hf_F_to_alpha = lambda F, d: (F*(d**2)-1)/(d**2-1)

dim = 4
k_list = [2, 3, 4]
F_list = np.linspace(0, 1, 20)
alpha_list = [hf_F_to_alpha(F, dim) for F in F_list]

ret_list = [[] for _ in k_list]
for k in k_list:
    witness = optimal_witness_isotropic(dim, k)
    for alpha in alpha_list:
        rho = numqi.state.Isotropic(dim, alpha)
        ret = np.real(np.trace(rho @ witness))
        ret_list[k-2].append(ret)

fig, ax = plt.subplots()
ax.plot(F_list, ret_list[0], label=r'$k=2$', marker='o')
ax.plot(F_list, ret_list[1], label=r'$k=3$', marker='o')
ax.plot(F_list, ret_list[2], label=r'$k=4$', marker='o')
ax.axhline(0, color='k', linestyle='dashed')
ax.text(0.8, 0.3, 'SEP', ha='center', va='center', size=12)
ax.text(0.2, -0.3, 'ENT', ha='center', va='center', size=12)
ax.set_xlabel(r'$F$')
ax.set_ylabel(r'$\text{Tr}[\rho W_k]$')
ax.legend()

# Bound entangled state

In [ ]:
rho_bes = numqi.entangle.load_upb('tiles', return_bes=True)[1]

model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=18, dtype=torch.complex128)
model.set_target_rho(rho_bes)
theta_optim = numqi.optimize.minimize(model, num_repeat=30, tol=1e-14, print_every_round=0).fun
distance_squared, info_best = model(return_info=True)
print(distance_squared)
# Gilbert Algorithm gives 0.002177, our algorithm gives 0.0019

model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=10, dtype=torch.float64)
model.set_target_rho(rho_bes)
theta_optim = numqi.optimize.minimize(model, num_repeat=30, tol=1e-14, print_every_round=0)
info = model(return_info=True)[1]

if abs(info['distance']-info_best['distance'])< 1e-11:
    tmp0 = model.manifold_ensemble_coeff().detach().numpy()
    tmp1 = [x().detach().numpy() for x in model.manifold_psi]
    tmp1 = [x*np.sign(x[:,:1]) for x in tmp1]
    ind0 = np.argsort(tmp0)
    coeff_p = tmp0[ind0]
    coeff_psi = [x[ind0] for x in tmp1]
    coeff_psi_pretty = np.concat([coeff_psi[0], 0*coeff_psi[0][:,:1], coeff_psi[1]], axis=1)
    print(coeff_p)
    print(coeff_psi_pretty)
else:
    print(abs(info['distance']-info_best['distance']))

In [ ]:
tmp = 0.1131000104906**2
print(tmp)
fraction = Fraction(tmp).limit_denominator(1000)
print(fraction)
print(float(fraction))

In [ ]:
rho_bes = numqi.entangle.load_upb('tiles', return_bes=True)[1]
distance_list = []
sigma_list = []
witness_list = []
p_list = np.linspace(0.9, 1, 20)
model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=18, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_bes, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])
    sigma_list.append(info['sigma'])
    witness_list.append(info['witness'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o')
ax.axvline(0.8908, color='k', linestyle='dashed')
# ax.text(0.9, 0.03, r'$p_{OSD}=0.8908$')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.set_title('BES from Tiles UPB')

In [ ]:
model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=18, dtype=torch.complex128)
model.set_target_rho(horodecki_state(5))
theta_optim = numqi.optimize.minimize(model, num_repeat=10, tol=1e-14, print_every_round=0)
info = model(return_info=True)[1]
print(info['distance'])
# b=0 0.11310001049064643
# b=5 0.11310001049061419

In [ ]:
model = HilbertSchmidtMeasure(dim_list=[3,3], rank=1, num_ensemble=18, dtype=torch.complex128)
b_list = np.linspace(0, 5, 100)
distance_list = []
for b in tqdm(b_list):
    rho = horodecki_state(b)
    model.set_target_rho(rho)
    theta_optim = numqi.optimize.minimize(model, num_repeat=10, tol=1e-14, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(b_list, distance_list)
ax.fill_between([0, 1], 0, 1, color='red', alpha=0.3, transform=ax.get_xaxis_transform(), label='NPT')
ax.fill_between([4, 5], 0, 1, color='red', alpha=0.3, transform=ax.get_xaxis_transform())
ax.fill_between([1, 4], 0, 1, color='yellow', alpha=0.3, transform=ax.get_xaxis_transform(), label='PPT')
ax.axhline(0, color='k', linestyle='dashed')
ax.set_xlabel(r'$b$')
ax.set_ylabel(r'HS measure')
ax.legend()
ax.set_title('Horodecki State')

In [ ]:
dim_list = [5,5]
rho = get_bound_entangled_state('grid-553')
model1 = HilbertSchmidtMeasure(dim_list=dim_list, rank=1, num_ensemble=2*5**2, dtype=torch.complex128)
model2 = HilbertSchmidtMeasure(dim_list=dim_list, rank=2, num_ensemble=2*5**2, dtype=torch.complex128)
# Distance to S_1
model1.set_target_rho(rho)
theta_optim1 = numqi.optimize.minimize(model1, num_repeat=10, tol=1e-14, print_every_round=0)
info1 = model1(return_info=True)[1]
print('rank=1:', info1['distance'])
model2.set_target_rho(rho)
theta_optim2 = numqi.optimize.minimize(model2, num_repeat=10, tol=1e-14, print_every_round=0)
info2 = model2(return_info=True)[1]
print('rank=2:', info2['distance'])
# rank=1: 0.048796577524112754
# rank=2: 0.003005054495800431

# Entangled state with larger Schmidt number

In [ ]:
def tensor(vec1, vec2):
    return np.kron(vec1, vec2)

def unfaithful_example_state(p):
    d = 4
    basis = [np.eye(d)[i] for i in range(d)]

    psi3 = (np.kron(basis[0], basis[0]) +
            np.kron(basis[1], basis[1]) +
            np.kron(basis[2], basis[2])) / np.sqrt(3)

    psi23 = (np.kron(basis[2], basis[3]) + np.kron(basis[3], basis[2])) / np.sqrt(2)

    rho_psi3 = np.outer(psi3, psi3.conjugate())
    rho_psi23 = np.outer(psi23, psi23.conjugate())

    rho = (1-p) * rho_psi3 + p * rho_psi23
    return rho

def two_isotropic(p):
    d = 2
    basis = [np.eye(d)[i] for i in range(d)]
    tmp0 = (np.kron(basis[0], basis[0]) +
            np.kron(basis[1], basis[1])) / np.sqrt(2)
    tmp1 = np.kron(tmp0, tmp0).reshape(2,2,2,2).transpose(0,2,1,3).reshape(4,4)
    rho = (1-p)* np.outer(tmp1, tmp1.conjugate()) + p * np.kron(np.eye(4), np.eye(4))/16
    return rho

In [ ]:
distance1_list = []
distance2_list = []
p_list = np.linspace(0, 1, 20)
model1 = HilbertSchmidtMeasure(dim_list=[4,4], rank=1, num_ensemble=2*4**2, dtype=torch.complex128)
model2 = HilbertSchmidtMeasure(dim_list=[4,4], rank=2, num_ensemble=2*4**2, dtype=torch.complex128)

for p in tqdm(p_list):
    rho = unfaithful_example_state(p)
    model1.set_target_rho(rho)
    theta_optim1 = numqi.optimize.minimize(model1, num_repeat=3, tol=1e-14, print_every_round=0)
    info1 = model1(return_info=True)[1]
    distance1_list.append(info1['distance'])

    model2.set_target_rho(rho)
    theta_optim2 = numqi.optimize.minimize(model2, num_repeat=3, tol=1e-14, print_every_round=0)
    info2 = model2(return_info=True)[1]
    distance2_list.append(info2['distance'])


In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance1_list, marker='o', label='k=2')
ax.plot(p_list, distance2_list, marker='o', label='k=3')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.legend()

In [ ]:
distance1_list = []
distance2_list = []
p_list = np.linspace(0, 1, 20)
model1 = HilbertSchmidtMeasure(dim_list=[4,4], rank=1, num_ensemble=2*4**2, dtype=torch.complex128)
model2 = HilbertSchmidtMeasure(dim_list=[4,4], rank=2, num_ensemble=2*4**2, dtype=torch.complex128)

for p in tqdm(p_list):
    rho = two_isotropic(p)
    model1.set_target_rho(rho)
    theta_optim1 = numqi.optimize.minimize(model1, num_repeat=3, tol=1e-14, print_every_round=0)
    info1 = model1(return_info=True)[1]
    distance1_list.append(info1['distance'])

    model2.set_target_rho(rho)
    theta_optim2 = numqi.optimize.minimize(model2, num_repeat=3, tol=1e-14, print_every_round=0)
    info2 = model2(return_info=True)[1]
    distance2_list.append(info2['distance'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance1_list, marker='o', label='k=2')
ax.plot(p_list, distance2_list, marker='o', label='k=3')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.legend()

In [ ]:
distance1_list = []
distance2_list = []
p_list = np.linspace(0, 1, 20)
model1 = HilbertSchmidtMeasure(dim_list=[4,4], rank=1, num_ensemble=2*16, dtype=torch.complex128)
model2 = HilbertSchmidtMeasure(dim_list=[4,4], rank=2, num_ensemble=2*16, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho, p)
    model1.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model1, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model1(return_info=True)[1]
    distance1_list.append(info['distance'])
    model2.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model2, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model2(return_info=True)[1]
    distance2_list.append(info['distance'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance1_list, marker='o', label='k=2')
ax.plot(p_list, distance2_list, marker='o', label='k=3')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.legend()

# Genuinely entangled state

In [ ]:
# load data from matlab file
import scipy.io as sio
data = sio.loadmat('stateexpre2.mat')
rho_list = data['exprhostate']

In [ ]:
W3 = numqi.state.W(3)
rho_W3 = W3[:, np.newaxis] @ W3[np.newaxis, :]

distance_list = []
witness_list = []
sigma_list = []
p_list = np.linspace(0, 1, 10)
model = GenuineHilbertSchmidtMeasure(dim_list=[2,2,2], num_ensemble=2*8, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_W3, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])
    witness_list.append(info['witness'])
    sigma_list.append(info['sigma'])

In [ ]:
rho_W3_exp = np.array(rho_list[0][0])
rho_W3_exp = np.trace(rho_W3_exp.reshape(2, 8, 2, 8), axis1=0, axis2=2)
print(density_matrix_fidelity(rho_W3_exp, rho_W3))

p_list_exp = p_list[-5:]
witness_list_exp = witness_list[-5:]
exp_results = []
for p, witness in zip(p_list_exp, witness_list_exp):
    rho_dep_exp = depolarizing_channel(rho_W3_exp, p)
    result = -np.real(np.trace(rho_dep_exp @ witness))
    exp_results.append(result)

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o', label='Numerical')
ax.plot(p_list_exp, exp_results, marker='x', color='red', label='Experiment')
ax.axvline(0.556, color='k', linestyle='dashed')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.text(0.6, 0.3, r'$p_{OSD}=0.556$')
ax.set_title(r'$W_3$')

In [ ]:
H3 = np.array([1, 0, 1, 0, 1, 0, 0, 1])/2
rho_H3 = H3[:, np.newaxis] @ H3[np.newaxis, :]
distance_list = []
sigma_list = []
witness_list = []
p_list = np.linspace(0, 1, 10)
model = GenuineHilbertSchmidtMeasure(dim_list=[2,2,2], num_ensemble=2*8, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_H3, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])
    sigma_list.append(info['sigma'])
    witness_list.append(info['witness'])

In [ ]:
rho_H3_exp = np.array(rho_list[0][1])
rho_H3_exp = np.trace(rho_H3_exp.reshape(2, 8, 2, 8), axis1=0, axis2=2)
print(density_matrix_fidelity(rho_H3_exp, rho_H3))


p_list_exp = p_list[-5:]
witness_list_exp = witness_list[-5:]
exp_results = []
for p, witness in zip(p_list_exp, witness_list_exp):
    rho_dep_exp = depolarizing_channel(rho_H3_exp, p)
    result = -np.real(np.trace(rho_dep_exp @ witness))
    exp_results.append(result)

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o', label='Numerical')
ax.plot(p_list_exp, exp_results, marker='x', color='red', label='Experiment')
ax.axvline(0.545, color='k', linestyle='dashed')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.text(0.6, 0.3, r'$p_{OSD}=0.545$')
ax.set_title(r'$H_3$')
ax.legend()

In [ ]:
W4_state = numqi.state.W(4)
rho_W4 = W4_state[:, np.newaxis] @ W4_state[np.newaxis, :]
distance_list = []
sigma_list = []
witness_list = []
p_list = np.linspace(0, 1, 10)
model = GenuineHilbertSchmidtMeasure(dim_list=[2,2,2,2], num_ensemble=2*16, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_W4, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])
    sigma_list.append(info['sigma'])
    witness_list.append(info['witness'])

In [ ]:
rho_W4_exp = np.array(rho_list[0][2])
print(density_matrix_fidelity(rho_W4_exp, rho_W4))

p_list_exp = p_list[-5:]
witness_list_exp = witness_list[-5:]
exp_results = []
for p, witness in zip(p_list_exp, witness_list_exp):
    rho_dep_exp = depolarizing_channel(rho_W4_exp, p)
    result = -np.real(np.trace(rho_dep_exp @ witness))
    exp_results.append(result)

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o', label='Numerical')
ax.plot(p_list_exp, exp_results, marker='x', color='red', label='Experiment')
ax.axvline(0.545, color='k', linestyle='dashed')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.text(0.6, 0.2, r'$p_{OSD}=0.545$')
ax.set_title(r'$W_4$')
ax.legend()

In [ ]:
dicke = numqi.state.Dicke(2,2)
rho_dicke = dicke[:, np.newaxis] @ dicke[np.newaxis, :]
distance_list = []
sigma_list = []
witness_list = []
p_list = np.linspace(0, 1, 10)
model = GenuineHilbertSchmidtMeasure(dim_list=[2,2,2,2], num_ensemble=2*16, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_dicke, p)
    model.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])
    sigma_list.append(info['sigma'])
    witness_list.append(info['witness'])

In [ ]:
rho_dicke_exp = np.array(rho_list[0][3])
print(density_matrix_fidelity(rho_dicke_exp, rho_dicke))

p_list_exp = p_list[-5:]
witness_list_exp = witness_list[-5:]
exp_results = []
for p, witness in zip(p_list_exp, witness_list_exp):
    rho_dep_exp = depolarizing_channel(rho_dicke_exp, p)
    result = -np.real(np.trace(rho_dep_exp @ witness))
    exp_results.append(result)

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o', label='Numerical')
ax.plot(p_list_exp, exp_results, marker='x', color='red', label='Experiment')
ax.axvline(0.540, color='k', linestyle='dashed')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.text(0.6, 0.3, r'$p_{OSD}=0.540$')
ax.set_title(r'$D_4^{2}$')

# Completely entangled states

In [ ]:
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)  # Pauli-X
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)  # Pauli-Y
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)  # Pauli-Z
identity = np.eye(2, dtype=complex)  #  I

def tensor_product(*matrices):
    result = matrices[0]
    for matrix in matrices[1:]:
        result = np.kron(result, matrix)
    return result

In [ ]:
identity_4 = tensor_product(identity, identity, identity, identity)
sigma_x_4 = tensor_product(sigma_x, sigma_x, sigma_x, sigma_x)
sigma_y_4 = tensor_product(sigma_y, sigma_y, sigma_y, sigma_y)
sigma_z_4 = tensor_product(sigma_z, sigma_z, sigma_z, sigma_z)
rho_S = (1/16) * (identity_4 + sigma_x_4 + sigma_y_4 + sigma_z_4)

In [ ]:
distance1_list = []
distance2_list = []
distance3_list = []
p_list = np.linspace(0, 1, 20)
model1 = HilbertSchmidtMeasure(dim_list=[2,2,2,2], rank=1, num_ensemble=4*16, dtype=torch.complex128)
model2 = HilbertSchmidtMeasure(dim_list=[2,8], rank=1, num_ensemble=4*16, dtype=torch.complex128)
model3 = HilbertSchmidtMeasure(dim_list=[2,2,4], rank=1, num_ensemble=4*16, dtype=torch.complex128)
for p in tqdm(p_list):
    rho_dep = depolarizing_channel(rho_S, p)
    model1.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model1, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model1(return_info=True)[1]
    distance1_list.append(info['distance'])
    model2.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model2, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model2(return_info=True)[1]
    distance2_list.append(info['distance'])
    model3.set_target_rho(rho_dep)
    theta_optim = numqi.optimize.minimize(model3, num_repeat=3, tol=1e-14, print_every_round=0)
    info = model3(return_info=True)[1]
    distance3_list.append(info['distance'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance1_list, marker='o', label='1|2|3|4')
ax.plot(p_list, distance2_list, marker='x', label='1|234')
ax.plot(p_list, distance3_list, marker='^', label='1|2|34')
ax.axvline(1/3, color='k', linestyle='dashed')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'HS measure')
ax.set_title(r'Smolin state')
ax.legend()
# log scale
# ax.set_yscale('log')

In [ ]:
def ghz_001_mixture(p):
    ghz =numqi.state.GHZ(3)
    # 001 state
    psi_001 = np.array([0, 1, 0, 0, 0, 0, 0, 0])
    rho_001 = psi_001[:, np.newaxis] @ psi_001[np.newaxis, :]
    rho_ghz = ghz[:, np.newaxis] @ ghz[np.newaxis, :]
    return (1-p)*rho_ghz + p*rho_001

In [ ]:
distance_list = []
p_list = np.linspace(0, 1, 10)
model = GenuineHilbertSchmidtMeasure(dim_list=[2,2,2], num_ensemble=2*16, distance='trace', dtype=torch.complex128)
for p in tqdm(p_list):
    rho = ghz_001_mixture(p)
    model.set_target_rho(rho)
    theta_optim = numqi.optimize.minimize(model, num_repeat=3, tol=1e-10, print_every_round=0)
    info = model(return_info=True)[1]
    distance_list.append(info['distance'])

In [ ]:
fig, ax = plt.subplots()
ax.plot(p_list, distance_list, marker='o')
ax.set_xlabel(r'$p$')
ax.set_ylabel(r'trace distance entanglement')
ax.set_title('GHZ-001 mixture')